# OBD Data Preparation

## Paths

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np

CSV_PATH = Path(r"C:\Users\olgap\OneDrive\MIPT_Master\04_Generation_NLP\RAG_for_material_search\data\OBD_2024_I_2026-02-19T12_56_13.csv")
OUT_DIR = Path(r"C:\Users\olgap\OneDrive\MIPT_Master\04_Generation_NLP\RAG_for_material_search\data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH, OUT_DIR

(WindowsPath('C:/Users/olgap/OneDrive/MIPT_Master/04_Generation_NLP/RAG_for_material_search/data/OBD_2024_I_2026-02-19T12_56_13.csv'),
 WindowsPath('C:/Users/olgap/OneDrive/MIPT_Master/04_Generation_NLP/RAG_for_material_search/data'))

## Load Raw Data

In [16]:
df_raw = pd.read_csv(CSV_PATH, sep=";", encoding="cp1252", low_memory=False)
print(df_raw.shape)
df_raw.head(1)

(26431, 100)


,UUID,Version,Name (de),Name (en),Kategorie (original),Kategorie (en),Konformitaet,Hintergrunddatenbank(en),Laenderkennung,Typ,...,TMR_FOSSILE,TMR_METALS,TMR_MINERALS,TMR_FORESTRY,TMR_AGRICULTURE,TMR_FISHERIES,TMR_ABIOTIC_SUBTOTAL,TMR_BIOTIC_SUBTOTAL,TMR_TOTAL,Unnamed: 99
0,c93da4c3-94c9-4c86-b092-610cf1cf012f,00.10.000,FOAMGLAS® T4+,FOAMGLAS® T4+,'Dämmstoffe' / 'Schaumglas' / 'Platten','Insulation materials' / 'Foam glass' / 'Panels','DIN EN 15804+A2' / 'ISO 14025','GaBi database (28d74cc0-db8b-4d7e-bc44-5f6d56...,RER,specific dataset,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Clean and Normalize

In [17]:
df = df_raw[df_raw["Typ"] != "template dataset"].copy()
df = df.drop(columns=["Unnamed: 99"], errors="ignore")

indicator_cols = [
    c for c in [
        "GWP", "ODP", "POCP", "AP", "EP", "ADPE", "ADPF",
        "PERE", "PERM", "PERT", "PENRE", "PENRM", "PENRT",
        "AP (A2)", "GWPtotal (A2)", "GWPbiogenic (A2)", "GWPfossil (A2)", "GWPluluc (A2)",
        "ETPfw (A2)", "PM (A2)", "EPmarine (A2)", "EPfreshwater (A2)", "EPterrestrial (A2)",
        "HTPc (A2)", "HTPnc (A2)", "IRP (A2)", "SOP (A2)", "ODP (A2)", "POCP (A2)",
        "ADPF (A2)", "ADPE (A2)", "WDP (A2)"
    ] if c in df.columns
]

physical_cols = [
    c for c in [
        "Schuettdichte (kg/m3)", "Flaechengewicht (kg/m2)", "Rohdichte (kg/m3)",
        "Schichtdicke (m)", "Ergiebigkeit (m2)", "Laengengewicht (kg/m)",
        "Stueckgewicht (kg)", "Umrechungsfaktor auf 1kg",
        "biogener Kohlenstoffgehalt in kg", "biogener Kohlenstoffgehalt (Verpackung) in kg"
    ] if c in df.columns
]

def to_float(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")

for c in indicator_cols + physical_cols:
    df[c] = to_float(df[c])

print(df.shape, "unique UUID:", df["UUID"].nunique())

(25872, 99) unique UUID: 2966


## Aggregate to Material Level

In [18]:
meta_cols = [
    c for c in [
        "Name (en)", "Name (de)", "Kategorie (en)", "Kategorie (original)",
        "Konformitaet", "Laenderkennung", "Typ", "Declaration owner",
        "Bezugsgroesse", "Bezugseinheit", "Referenzjahr", "Gueltig bis", "URL"
    ] if c in df.columns
]

df_meta = df.groupby("UUID").first()[meta_cols + physical_cols].reset_index()
df_a1a3 = df[df["Modul"] == "A1-A3"].groupby("UUID").first()[indicator_cols].reset_index()
df_materials = df_meta.merge(df_a1a3, on="UUID", how="left")

gwp_col = "GWPtotal (A2)" if "GWPtotal (A2)" in df_materials.columns else "GWP"
df_materials = df_materials[df_materials[gwp_col].notna()].reset_index(drop=True)
print(df_materials.shape, "unique UUID:", df_materials["UUID"].nunique())

(2491, 56) unique UUID: 2491


## Build Text and Numeric Tables

In [19]:
indicator_fullnames = {
    "GWPtotal (A2)": ("Global Warming Potential — Total", "kg CO2-eq"),
    "GWPfossil (A2)": ("Global Warming Potential — Fossil", "kg CO2-eq"),
    "AP (A2)": ("Acidification Potential", "mol H+-eq"),
    "ODP (A2)": ("Ozone Depletion Potential", "kg CFC-11-eq"),
    "EPfreshwater (A2)": ("Eutrophication Potential — Freshwater", "kg P-eq"),
    "PENRT": ("Primary Energy Non-Renewable — Total", "MJ"),
    "PERT": ("Primary Energy Renewable — Total", "MJ"),
    "GWP": ("Global Warming Potential", "kg CO2-eq")
}

priority_indicators = [c for c in [
    "GWPtotal (A2)", "GWPfossil (A2)", "AP (A2)", "ODP (A2)", "EPfreshwater (A2)", "PENRT", "PERT", "GWP"
] if c in df_materials.columns]

def build_text_document(row):
    parts = []
    name = row.get("Name (en)") if pd.notna(row.get("Name (en)")) else row.get("Name (de)", "Unknown product")
    parts.append(f"Product: {name}.")
    if pd.notna(row.get("Kategorie (en)")):
        parts.append(f"Category: {row.get('Kategorie (en)')}.")
    if pd.notna(row.get("Bezugsgroesse")) and pd.notna(row.get("Bezugseinheit")):
        parts.append(f"Declared unit: {row.get('Bezugsgroesse')} {row.get('Bezugseinheit')}.")
    if pd.notna(row.get("Konformitaet")):
        parts.append(f"Standard: {row.get('Konformitaet')}.")
    if pd.notna(row.get("Laenderkennung")):
        parts.append(f"Country: {row.get('Laenderkennung')}.")
    if pd.notna(row.get("Declaration owner")):
        parts.append(f"Declaration owner: {row.get('Declaration owner')}.")

    impacts = []
    for col in priority_indicators:
        val = row.get(col)
        if pd.notna(val):
            full_name, unit = indicator_fullnames.get(col, (col, ""))
            impacts.append(f"{full_name} ({col}): {val} {unit}".strip())
    if impacts:
        parts.append("Environmental indicators (A1-A3): " + "; ".join(impacts) + ".")
    return " ".join(parts)

df_text = pd.DataFrame({
    "UUID": df_materials["UUID"],
    "text_document": df_materials.apply(build_text_document, axis=1),
    "dataset_type": df_materials["Typ"] if "Typ" in df_materials.columns else ""
})

numeric_keep = [c for c in [
    "UUID", "Name (en)", "Kategorie (en)", "Typ", "Bezugsgroesse", "Bezugseinheit", "Laenderkennung", "Declaration owner"
] if c in df_materials.columns] + indicator_cols + physical_cols
df_numeric = df_materials[numeric_keep].copy()

print("df_text:", df_text.shape, "df_numeric:", df_numeric.shape)

df_text: (2491, 3) df_numeric: (2491, 50)


## Build Evaluation Labels and Qrels

In [20]:
_df = df_text.merge(df_numeric[[c for c in ["UUID", "Name (en)", "Kategorie (en)"] if c in df_numeric.columns]], on="UUID", how="left")
_df = _df.rename(columns={"Name (en)": "product", "Kategorie (en)": "category"})

SPECS = {
    "q1_concrete": {
        "query": "ready mixed concrete",
        "cat_regex": r"Mineral building products.*Mortar and Concrete.*Ready mixed concrete",
        "prod_regex": r"\bready[- ]mixed concrete\b"
    },
    "q2_glass_wool_insulation": {
        "query": "glass wool insulation",
        "cat_regex": r"Insulation materials.*(Glass wool|Mineral wool|Rock wool)",
        "prod_regex": r"(glass wool|mineral wool|rock wool)"
    },
    "q3_roofing_membrane": {
        "query": "roofing membrane",
        "cat_regex": r"Plastics.*Roofing membranes",
        "prod_regex": r"(roofing membrane|tpo|pvc sheet)"
    }
}

labels = {u: [] for u in _df["UUID"]}
for qid, spec in SPECS.items():
    cat_match = _df["category"].fillna("").str.contains(spec["cat_regex"], case=False, regex=True)
    prod_match = _df["product"].fillna("").str.contains(spec["prod_regex"], case=False, regex=True)
    matched = _df.loc[cat_match | prod_match, "UUID"].unique()
    for u in matched:
        labels[u].append(qid)

df_text["eval_labels"] = df_text["UUID"].map(lambda u: "|".join(labels.get(u, [])))

queries = pd.DataFrame([{"query_id": qid, "query_text": spec["query"]} for qid, spec in SPECS.items()])
qrels = (
    df_text.loc[df_text["eval_labels"].ne(""), ["UUID", "eval_labels"]]
    .assign(query_id=lambda x: x["eval_labels"].str.split("|"))
    .explode("query_id")
    .drop(columns=["eval_labels"])
    .assign(relevance=1)
    .drop_duplicates(["query_id", "UUID"])
    .sort_values(["query_id", "UUID"])
    .reset_index(drop=True)
)

print(df_text["eval_labels"].value_counts().head())
print(qrels.groupby("query_id")["UUID"].nunique())

eval_labels
                            2347
q1_concrete                   81
q3_roofing_membrane           33
q2_glass_wool_insulation      30
Name: count, dtype: int64
query_id
q1_concrete                 81
q2_glass_wool_insulation    30
q3_roofing_membrane         33
Name: UUID, dtype: int64


C:\Users\olgap\AppData\Local\Temp\ipykernel_1312\800566727.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  cat_match = _df["category"].fillna("").str.contains(spec["cat_regex"], case=False, regex=True)
C:\Users\olgap\AppData\Local\Temp\ipykernel_1312\800566727.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  prod_match = _df["product"].fillna("").str.contains(spec["prod_regex"], case=False, regex=True)


## Save Prepared Files

In [21]:
df_text.to_csv(OUT_DIR / "df_text.csv", index=False, encoding="utf-8-sig")
df_numeric.to_csv(OUT_DIR / "df_numeric.csv", index=False, encoding="utf-8-sig")
queries.to_csv(OUT_DIR / "eval_queries.csv", index=False, encoding="utf-8-sig")
qrels.to_csv(OUT_DIR / "eval_qrels.csv", index=False, encoding="utf-8-sig")

print("Saved:")
print(OUT_DIR / "df_text.csv")
print(OUT_DIR / "df_numeric.csv")
print(OUT_DIR / "eval_queries.csv")
print(OUT_DIR / "eval_qrels.csv")

Saved:
C:\Users\olgap\OneDrive\MIPT_Master\04_Generation_NLP\RAG_for_material_search\data\df_text.csv
C:\Users\olgap\OneDrive\MIPT_Master\04_Generation_NLP\RAG_for_material_search\data\df_numeric.csv
C:\Users\olgap\OneDrive\MIPT_Master\04_Generation_NLP\RAG_for_material_search\data\eval_queries.csv
C:\Users\olgap\OneDrive\MIPT_Master\04_Generation_NLP\RAG_for_material_search\data\eval_qrels.csv
